In [ ]:
import os 
#os.chdir("..")

In [2]:
from src.data import preprocess_dataset
from src.data.punctuation_dataset import FINAL_LABELS, INITIAL_LABELS
from src.data.capitalization_dataset import TITLE, MIXED, UPPERCASE

/home/sebastian/personal/uba/Aprendizaje-Automatico-UBA/tp/TP2-Aprendizaje-Automatico-UBA/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
punctuation, capitalization = preprocess_dataset()

In [12]:
punctuation.len

402525

In [29]:
import numpy as np

np.mean([len(x) for x in punctuation.data])

np.float64(38.46812496118253)

In [9]:
def formatPunctuation(x, y):
    text = ""
    for i, token in enumerate(x):
        if not token.startswith("##"):
            text += " "
        token = token.removeprefix("##")
        text += INITIAL_LABELS[y[i][0]] + token + FINAL_LABELS[y[i][1]]
    return text.strip()

In [10]:
i = 4
x, y = punctuation[i]
print("input: ", punctuation.data[i])
print("target: ", punctuation.data[i])
print("tokenización: ", x)
print("format(model(tokenizacion)): ", formatPunctuation(x,y))

input:  No tienen absolutamente nada en común.
target:  No tienen absolutamente nada en común.
tokenización:  ['no', 'tienen', 'absolutam', '##ente', 'nada', 'en', 'común']
format(model(tokenizacion)):  no tienen absolutamente nada en común.


In [11]:
y.shape

torch.Size([7, 2])

In [7]:
def formatCapitalization(x, y):
    text = ""
    for i, token in enumerate(x):
        is_continuation = token.startswith("##")
        if not is_continuation:
            text += " "
        token = token.removeprefix("##")
        cap = int(y[i])
        starts_mixed = cap == MIXED and (i == 0 or int(y[i - 1]) != MIXED)
        if cap == UPPERCASE:
            token = token.upper()
        elif (cap == TITLE and not is_continuation) or starts_mixed:
            # Para MIXED no sabemos qué letras van en mayúscula: solo ponemos la primera
            token = token.capitalize()
        text += token
    return text.strip()

In [8]:
i = 4
x, y = capitalization[i]

print("input: ", capitalization.data[i])
print("target: ", capitalization.data[i])
print("tokenización: ", x)
print("format(model(tokenizacion)): ", formatCapitalization(x,y))

input:  No tienen absolutamente nada en común.
target:  No tienen absolutamente nada en común.
tokenización:  ['no', 'tienen', 'absolutam', '##ente', 'nada', 'en', 'común']
format(model(tokenizacion)):  No tienen absolutamente nada en común


In [27]:
text_orig = "¿Cuándo vamos a McDonald's? Ellos no vienen hoy. ¿Dónde están ahora?"

# Igual que TextDataset.__getitem__: se trabaja con la lista de palabras, no con el string
words = text_orig.split()
word_tokens = punctuation._tokenize_words(words)
tokens = [token for tokens in word_tokens for token in tokens]
y_punt = punctuation._expected_output(words, word_tokens)
y_cap = capitalization._expected_output(words, word_tokens)

print(f"{'token':10} {'inicial':7} {'final':5} cap")
for token, (ini, fin), cap in zip(tokens, y_punt.tolist(), y_cap.tolist()):
    print(f"{token:10} {INITIAL_LABELS[ini]:7} {FINAL_LABELS[fin]:5} {cap}")

print()
print("format punt: ", formatPunctuation(tokens, y_punt))
print("format cap:  ", formatCapitalization(tokens, y_cap))

token      inicial final cap
cu         ¿             1
##ánd                    1
##o                      1
va                       0
##mos                    0
a                        0
m                        2
##c                      2
##dona                   2
##ld                     2
'                        2
s                  ?     2
ellos                    1
no                       0
viene                    0
##n                      0
hoy                .     0
dó         ¿             1
##nde                    1
están                    0
ahora              ?     0

format punt:  ¿cuándo vamos a mcdonald ' s? ellos no vienen hoy. ¿dónde están ahora?
format cap:   Cuándo vamos a Mcdonald ' s Ellos no vienen hoy Dónde están ahora
